In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "data" / "raw").exists() and root != root.parent:
    root = root.parent
raw_dir = root / "data" / "raw"

frames = []
for path in sorted(raw_dir.glob("[0-9][0-9][0-9][0-9].csv")):
    d = pd.read_csv(path)
    d["file_year"] = int(path.stem)
    frames.append(d)

raw_all = pd.concat(frames, ignore_index=True)
print(len(frames), "files loaded", raw_all.shape)

In [ ]:
#filtering the rows 
clean = raw_all.copy()
print("start:", len(clean))

clean = clean[~clean["tourney_level"].isin(["D", "O"])]
print("after Davis Cup and Olympics:", len(clean))

keep_a = ["Doha", "Los Cabos"]
clean = clean[(clean["tourney_level"] != "A") | clean["tourney_name"].isin(keep_a)]
print("after team events and Next Gen:", len(clean))

clean = clean[~clean["score"].str.contains("W/O|DEF", na=False)]
print("after walkovers and defaults:", len(clean))

In [ ]:
#cleaning 
clean = clean.copy()
clean["indoor"] = clean["indoor"].replace({"0": "O", 0: "O"})
clean["date"] = pd.to_datetime(clean["tourney_date"].astype(str), format="%Y%m%d")
clean["retired"] = clean["score"].str.contains("RET", na=False)
clean["event_no"] = clean["tourney_id"].astype(str).str.split("-").str[1]

In [ ]:
#check cleaning
print(clean["indoor"].value_counts(dropna=False).to_string())
print("missing surface:", clean["surface"].isna().sum())
print("missing event_no:", clean["event_no"].isna().sum())
print("missing best_of:", clean["best_of"].isna().sum())
print(clean["date"].min(), clean["date"].max())

In [ ]:
# mark 14 tournaments we will use 
t = pd.read_csv(root / "data" / "own" / "tournaments.csv", dtype={"event_no": str})
clean["analysed"] = clean["event_no"].isin(t["event_no"]) & clean["tourney_level"].isin(["G", "M", "F"])

print(clean["analysed"].sum())
print(clean[clean["analysed"]].groupby("file_year").size().to_string())

In [ ]:
other = [c for c in clean.columns if c != "match_num"]
same = clean[clean.duplicated(subset=other, keep=False)]
print(len(same))
print(same.groupby(["tourney_level", "tourney_name"]).size().to_string())

In [ ]:
print("empty match_num:", raw_all["match_num"].isna().sum())
print(raw_all["indoor"].value_counts(dropna=False).to_string())
print(raw_all[raw_all["tourney_level"] == "A"]["tourney_name"].value_counts().to_string())

In [ ]:
key = ["tourney_id", "round", "winner_id", "loser_id"]
rep = clean[clean.duplicated(subset=key + ["file_year"], keep=False)]
print("repeated rows:", len(rep))
print(rep.groupby(["tourney_level", "tourney_name"]).size().to_string())

other = [c for c in clean.columns if c != "match_num"]
same = clean[clean.duplicated(subset=other, keep=False)]
print("identical apart from match_num:", len(same))

In [ ]:
rows = []
for y in range(2000, 2027):
    old = pd.read_csv(raw_dir / "before_tml_fix" / f"{y}.csv").astype(str)
    new = pd.read_csv(raw_dir / f"{y}.csv").astype(str)
    common = [c for c in new.columns if c in old.columns]
    only_new = (new[common].merge(old[common].drop_duplicates(), on=common, how="left", indicator=True)["_merge"] == "left_only").sum()
    only_old = (old[common].merge(new[common].drop_duplicates(), on=common, how="left", indicator=True)["_merge"] == "left_only").sum()
    rows.append((y, len(old), len(new), only_new, only_old, len(set(new.columns) ^ set(old.columns))))

print(pd.DataFrame(rows, columns=["year", "old_rows", "new_rows", "rows_only_in_new", "rows_only_in_old", "columns_differing"]).to_string(index=False))

In [ ]:
key = ["tourney_id", "round", "winner_id", "loser_id", "file_year"]
before = len(clean)
clean = clean.drop_duplicates(subset=[c for c in clean.columns if c != "match_num"], keep="first")
print("dropped:", before - len(clean))
print("repeats of the key left:", clean.duplicated(subset=key).sum())

In [ ]:
key = ["tourney_id", "round", "winner_id", "loser_id"]
for y in [2023, 2024, 2025, 2026]:
    old = pd.read_csv(raw_dir / "before_tml_fix" / f"{y}.csv").drop_duplicates(subset=key)
    new = pd.read_csv(raw_dir / f"{y}.csv").drop_duplicates(subset=key)
    m = old.merge(new, on=key, suffixes=("_old", "_new"), how="outer", indicator=True)
    print(y, m["_merge"].value_counts().to_dict())
    both = m[m["_merge"] == "both"]
    diffs = {}
    for c in new.columns:
        if c in key or c not in old.columns:
            continue
        a, b = both[c + "_old"], both[c + "_new"]
        n = (~((a == b) | (a.isna() & b.isna()))).sum()
        if n:
            diffs[c] = int(n)
    print(diffs)

In [ ]:
r = clean[(clean["tourney_name"] == "Rome Masters") & clean["file_year"].isin([2020, 2021, 2022])]
print(r.groupby(["file_year", "round"]).size().unstack().to_string())

In [ ]:
t = pd.read_csv(root / "data" / "own" / "tournaments.csv", dtype={"event_no": str})
clean["analysed"] = clean["event_no"].isin(t["event_no"]) & clean["tourney_level"].isin(["G", "M", "F"])
print(clean["analysed"].sum())
print(clean[clean["analysed"]].groupby("file_year").size().to_string())

In [ ]:
q = raw_all[(raw_all["tourney_name"] == "Rome Masters") & (raw_all["file_year"] == 2021) & (raw_all["round"] == "R32")]
print(q["score"].str.contains("W/O|DEF", na=False).sum())

In [ ]:

round_order = {"R128": 1, "R64": 2, "R32": 3, "R16": 4, "RR": 5, "QF": 6, "SF": 7, "F": 8}
clean["round_order"] = clean["round"].map(round_order)

print(clean["round"].value_counts(dropna=False).to_string())
print("rounds without an order:", clean["round_order"].isna().sum())

In [ ]:
clean = clean.sort_values(["date", "tourney_id", "round_order", "match_num", "winner_id"]).reset_index(drop=True)
clean["match_id"] = clean.index + 1

print(clean["match_id"].is_unique, clean["match_id"].min(), clean["match_id"].max())
print(len(clean))

In [ ]:
chk = clean.groupby("tourney_id")["match_id"].agg(["min", "max", "count"])
print((chk["max"] - chk["min"] + 1 == chk["count"]).mean())

In [ ]:
chk = clean.groupby("tourney_id")["match_id"].agg(["min", "max", "count"])
bad = chk[chk["max"] - chk["min"] + 1 != chk["count"]]
print(len(bad), "of", len(chk), "tournaments")

ids = bad.index[:10]
print(clean[clean["tourney_id"].isin(ids)].groupby(["tourney_id", "tourney_name"])["date"].nunique().to_string())

In [ ]:
rr = clean[clean["round"] == "RR"]
print(rr.groupby(["tourney_level", "tourney_name"])["file_year"].agg(["min", "max", "count"]).to_string())

In [ ]:
bad_ids = bad.index
sub = clean[clean["tourney_id"].isin(bad_ids)]
n_dates = sub.groupby("tourney_id")["date"].nunique()
year_of = sub.groupby("tourney_id")["file_year"].first()
print(n_dates.groupby(year_of).agg(["count", "min", "max"]).to_string())

print(sorted(clean[clean["tourney_id"] == "2019-418"]["date"].dt.strftime("%Y-%m-%d").unique()))
print(sorted(clean[clean["tourney_id"] == "2026-308"]["date"].dt.strftime("%Y-%m-%d").unique()))

rr_events = clean[clean["tourney_level"] == "250"].loc[lambda d: d["round"] == "RR", "tourney_id"].unique()
print(clean[clean["tourney_id"].isin(rr_events)].groupby("tourney_name")["round"].unique().to_string())

In [ ]:
clean["tourney_start"] = clean.groupby("tourney_id")["date"].transform("min")
clean = clean.sort_values(["tourney_start", "tourney_id", "round_order", "date", "match_num", "winner_id"]).reset_index(drop=True)
clean["match_id"] = clean.index + 1

chk = clean.groupby("tourney_id")["match_id"].agg(["min", "max", "count"])
print((chk["max"] - chk["min"] + 1 == chk["count"]).mean())
print(clean["match_id"].is_unique, len(clean))